In [1]:
import pandas as pd
import numpy as np

from utils import Display

In [2]:
housing = pd.read_csv('processed_data/01_housing.csv')
park = pd.read_csv('processed_data/01_park.csv')

In [7]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from sklearn.preprocessing import LabelEncoder,StandardScaler

dist_le = LabelEncoder()
neigh_le = LabelEncoder()

housing['dist_idx'] = dist_le.fit_transform(housing['区域'])
housing['neigh_idx'] = neigh_le.fit_transform(housing['板块'])
  
n_dists = len(dist_le.classes_)
n_neighs = len(neigh_le.classes_)


dist_tensor = torch.tensor(housing['dist_idx'].values, dtype=torch.long)
neigh_tensor = torch.tensor(housing['neigh_idx'].values, dtype=torch.long)
target_tensor = torch.tensor(housing['log_price'].values, dtype=torch.float32)

In [8]:
class EmbeddingModel(nn.Module):
    def __init__(self, n_dists, n_neighs, dist_dim=4, neigh_dim=16):
        super().__init__()
        self.dist_emb = nn.Embedding(n_dists, dist_dim)
        self.neigh_emb = nn.Embedding(n_neighs, neigh_dim)
        self.net = nn.Sequential(
            nn.Linear(dist_dim + neigh_dim, 128),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.Dropout(0.2),

            nn.Linear(128, 64),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.Dropout(0.1),
            
            nn.Linear(64, 1)
        )
        
    def forward(self, d_idx, n_idx):
        d = self.dist_emb(d_idx)
        n = self.neigh_emb(n_idx)
        out = torch.cat([d, n], dim=1)
        return self.net(out)
    


In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset, random_split
import copy 


full_dataset = TensorDataset(dist_tensor, neigh_tensor, target_tensor)

train_size = int(0.8 * len(full_dataset))
val_size = len(full_dataset) - train_size
train_dataset, val_dataset = random_split(full_dataset, [train_size, val_size])

train_loader = DataLoader(train_dataset, batch_size=256, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=256, shuffle=False)

model = EmbeddingModel(n_dists, n_neighs)
optimizer = optim.Adam(model.parameters(), lr=0.0005)
criterion = nn.MSELoss()

epochs = 200 
patience = 10 
best_val_loss = float('inf')
counter = 0
best_model_state = None 

for epoch in range(epochs):
    
    model.train()
    train_loss = 0
    for b_dist, b_neigh, b_target in train_loader:
        optimizer.zero_grad()
        preds = model(b_dist, b_neigh)
        loss = criterion(preds, b_target.view(-1, 1)) 
        loss.backward()
        optimizer.step()
        train_loss += loss.item()

    model.eval()
    val_loss = 0
    with torch.no_grad():
        for b_dist, b_neigh, b_target in val_loader:
            preds = model(b_dist, b_neigh)
            loss = criterion(preds, b_target.view(-1, 1))
            val_loss += loss.item()
    
    avg_train_loss = train_loss / len(train_loader)
    avg_val_loss = val_loss / len(val_loader)

    if avg_val_loss < best_val_loss:
        best_val_loss = avg_val_loss
        best_model_state = copy.deepcopy(model.state_dict())
        best_epoch = epoch + 1
        counter = 0
        print(f"Epoch {epoch+1}: New best model saved! (Val Loss: {avg_val_loss:.4f})")
    else:
        counter += 1
        if (epoch + 1) % 10 == 0:
            print(f"Epoch {epoch+1}: Train Loss: {avg_train_loss:.4f}, Val Loss: {avg_val_loss:.4f}")
        
    if counter >= patience:
        print(f"Early stopping triggered at epoch {epoch+1}. Best Val Loss: {best_val_loss:.4f}")
        break

if best_model_state:
    model.load_state_dict(best_model_state)
    print("Successfully loaded the best model parameters.")

Epoch 1: New best model saved! (Val Loss: 0.3471)
Epoch 2: New best model saved! (Val Loss: 0.3471)
Epoch 3: New best model saved! (Val Loss: 0.3262)
Epoch 4: New best model saved! (Val Loss: 0.3053)
Epoch 8: New best model saved! (Val Loss: 0.3041)
Epoch 10: New best model saved! (Val Loss: 0.2978)
Epoch 11: New best model saved! (Val Loss: 0.2907)
Epoch 14: New best model saved! (Val Loss: 0.2829)
Epoch 20: Train Loss: 0.3487, Val Loss: 0.2875


In [ ]:
dist_embeddings_raw = best_model_state['dist_emb.weight'].cpu().numpy()
neigh_embeddings_raw = best_model_state['neigh_emb.weight'].cpu().numpy()

print(f"区域 Embedding 形状: {dist_embeddings_raw.shape}") 
print(f"板块 Embedding 形状: {neigh_embeddings_raw.shape}") 

dist_col_names = [f'dist_v_{i}' for i in range(dist_embeddings_raw.shape[1])]
df_dist_vectors = pd.DataFrame(dist_embeddings_raw, columns=dist_col_names)
df_dist_vectors['区县'] = dist_le.classes_

neigh_col_names = [f'neigh_v_{i}' for i in range(neigh_embeddings_raw.shape[1])]
df_neigh_vectors = pd.DataFrame(neigh_embeddings_raw, columns=neigh_col_names)
df_neigh_vectors['板块'] = neigh_le.classes_

In [ ]:
housing = housing.merge(df_dist_vectors, left_on='区域', right_on='区域', how='left')
housing = housing.merge(df_neigh_vectors, left_on='板块', right_on='板块', how='left')
